In [7]:
#r "C:\Users\user\Desktop\Remish\practice2026\task17\bin\Debug\net10.0\task17.dll"
#r "nuget: ScottPlot, 5.0.21"

using System;
using System.Diagnostics;
using System.Linq;
using System.Threading;
using System.Collections.Generic;
using System.Collections.Concurrent;
using System.IO;
using ScottPlot;
using task17;
using Microsoft.AspNetCore.Html;

public class TestCommand : ICommand
{
    private int _counter = 0;
    private readonly int _id;
    
    public TestCommand(int id)
    {
        _id = id;
    }
    
    public void Execute()
    {
        Console.WriteLine($"Поток {_id} вызов {++_counter}");
        Thread.Sleep(5); 
    }
}

public class LongCommandWrapper : ILongCommand
{
    private readonly ICommand _inner;
    private int _remaining;
    
    public bool IsCompleted => _remaining <= 0;
    public string Name { get; }
    
    public LongCommandWrapper(ICommand command, int totalCalls, string name = "")
    {
        _inner = command ?? throw new ArgumentNullException(nameof(command));
        _remaining = totalCalls;
        Name = string.IsNullOrEmpty(name) ? command.GetType().Name : name;
    }
    
    public void Execute()
    {
        if (!IsCompleted)
        {
            _remaining--;
            _inner.Execute();
        }
    }
}

Console.WriteLine("ТЕСТ 2: Round Robin ");
var server2 = new ServerThread(new RoundRobinScheduler());
server2.Start();

for (int i = 1; i <= 5; i++)
{
    server2.Add(new LongCommandWrapper(new TestCommand(i), 3));
}

Thread.Sleep(500);
server2.HardStop();
server2.Join();
Console.WriteLine();


Console.WriteLine("ТЕСТ 3: Чередование");
var timeline = new List<string>();
var server3 = new ServerThread(new RoundRobinScheduler());
server3.Start();

for (int i = 1; i <= 5; i++)
{
    int id = i;
    var logCmd = new SimpleCommand(() => 
    {
        timeline.Add($"#{id}");
        Thread.Sleep(5);
    });
    server3.Add(new LongCommandWrapper(logCmd, 3, $"#{id}"));
}

Thread.Sleep(500);
server3.HardStop();
server3.Join();

Console.WriteLine("Порядок выполнения: " + string.Join(" -> ", timeline));
Console.WriteLine($"Ожидалось: #1 -> #2 -> #3 -> #4 -> #5 -> #1 -> #2 -> #3 -> #4 -> #5 -> #1 -> #2 -> #3 -> #4 -> #5");
Console.WriteLine();

Console.WriteLine("ТЕСТ 4: Сравнение времени выполнения");
var counts = new List<int>();
var sequentialTimes = new List<double>();
var roundRobinTimes = new List<double>();

for (int n = 1; n <= 10; n++)
{
    var sw1 = Stopwatch.StartNew();
    var srv1 = new ServerThread(new RoundRobinScheduler());
    int done1 = 0;
    srv1.Start();
    for (int i = 0; i < n; i++)
    {
        srv1.Add(new SimpleCommand(() =>
        {
            for (int s = 0; s < 10; s++)
            {
                Interlocked.Increment(ref done1);
                Thread.Sleep(5);
            }
        }));
    }
    while (done1 < n * 10) Thread.Sleep(1);
    srv1.HardStop();
    srv1.Join();
    sw1.Stop();

    var sw2 = Stopwatch.StartNew();
    var srv2 = new ServerThread(new RoundRobinScheduler());
    int done2 = 0;
    srv2.Start();
    for (int i = 0; i < n; i++)
    {
        var workCmd = new SimpleCommand(() =>
        {
            Interlocked.Increment(ref done2);
            Thread.Sleep(5);
        });
        srv2.Add(new LongCommandWrapper(workCmd, 10));
    }
    while (done2 < n * 10) Thread.Sleep(1);
    srv2.HardStop();
    srv2.Join();
    sw2.Stop();

    counts.Add(n);
    sequentialTimes.Add(sw1.Elapsed.TotalMilliseconds);
    roundRobinTimes.Add(sw2.Elapsed.TotalMilliseconds);
    Console.WriteLine($"Задач: {n,2} | Последовательно: {sw1.Elapsed.TotalMilliseconds,6:F0} мс | Round Robin: {sw2.Elapsed.TotalMilliseconds,6:F0} мс");
}

for (int i = 0; i < counts.Count; i++)
    lines.Add($"Задач: {counts[i],2} | Последовательно: {sequentialTimes[i],6:F0} мс | Round Robin: {roundRobinTimes[i],6:F0} мс");
File.WriteAllLines(logPath, lines);
Console.WriteLine($"\nРезультаты сохранены: {logPath}");

var plot = new Plot();
plot.Title("Время выполнения: последовательно vs Round Robin");
plot.XLabel("Количество задач");
plot.YLabel("Время (мс)");

plot.Add.Scatter(counts.Select(x => (double)x).ToArray(), sequentialTimes.ToArray())
    .Label = "Последовательно (без планировщика)";
plot.Add.Scatter(counts.Select(x => (double)x).ToArray(), roundRobinTimes.ToArray())
    .Label = "Round Robin (с планировщиком)";

plot.ShowLegend();

var fn = Path.Combine(Directory.GetCurrentDirectory(), "graoh.png");
plot.SavePng(fn, 800, 600);
Console.WriteLine($"График сохранён: {fn}");
display(HTML($"<img src='{fn}?t={DateTime.Now.Ticks}' width='700'/>"));

Installed Packages ScottPlot, 5.0.21

ТЕСТ 2: Round Robin 
Поток 1 вызов 1
Поток 2 вызов 1
Поток 1 вызов 2
Поток 3 вызов 1
Поток 2 вызов 2
Поток 4 вызов 1
Поток 1 вызов 3
Поток 5 вызов 1
Поток 3 вызов 2
Поток 2 вызов 3
Поток 4 вызов 2
Поток 5 вызов 2
Поток 3 вызов 3
Поток 4 вызов 3
Поток 5 вызов 3

ТЕСТ 3: Чередование
Порядок выполнения: #1 -> #2 -> #1 -> #3 -> #2 -> #4 -> #1 -> #5 -> #3 -> #2 -> #4 -> #5 -> #3 -> #4 -> #5
Ожидалось: #1 -> #2 -> #3 -> #4 -> #5 -> #1 -> #2 -> #3 -> #4 -> #5 -> #1 -> #2 -> #3 -> #4 -> #5

ТЕСТ 4: Сравнение времени выполнения
Задач:  1 | Последовательно:    137 мс | Round Robin:    136 мс
Задач:  2 | Последовательно:    276 мс | Round Robin:    306 мс
Задач:  3 | Последовательно:    433 мс | Round Robin:    450 мс
Задач:  4 | Последовательно:    557 мс | Round Robin:    589 мс
Задач:  5 | Последовательно:    615 мс | Round Robin:    687 мс
Задач:  6 | Последовательно:    872 мс | Round Robin:    911 мс
Задач:  7 | Последовательно:   1068 мс | Round Robin:   1070 мс
Задач:  8 | Последовательно